In [ ]:
#use "../lib/nb.ml";;
load_css ()

# A Shift-Reduce Parser for Arithmetic Expressions

In this notebook we implement a generic *shift reduce parser*.  The parse table that we use 
implements the following grammar for arithmetic expressions:
$$
  \begin{eqnarray*}
  \mathrm{expr}        & \rightarrow & \mathrm{expr}\;\;\texttt{'+'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{expr}\;\;\texttt{'-'}\;\;\mathrm{product}   \\
                       & \mid        & \mathrm{product}                                    \\[0.2cm]
  \mathrm{product}     & \rightarrow & \mathrm{product}\;\;\texttt{'*'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{product}\;\;\texttt{'/'}\;\;\mathrm{factor} \\
                       & \mid        & \mathrm{factor}                                     \\[0.2cm]
  \mathrm{factor}      & \rightarrow & \texttt{'('} \;\;\mathrm{expr} \;\;\texttt{')'}     \\
                       & \mid        & \texttt{NUMBER} 
  \end{eqnarray*}
$$

## Type Checking

The functions in this notebook carry *type annotations*.  In OCaml, these annotations are checked by the
compiler before a cell is executed, so there is no need for a separate type checker like *basedpyright*.

## Implementing a Scanner

In order to parse, we need a scanner.  We will use a scanner that is similar to the one that we have already used for our *top down parser* discussed earlier in Chapter 6.

In [ ]:
load_lib "regex.ml"

The function `tokenize` scans the string `s` into a list of tokens using regular expressions.  The scanner distinguishes between
* whitespace, which is discarded,
* numbers,
* arithmetical operators and parenthesis,
* all remaining characters, which are treated as lexical errors.

The regular expression has four groups, so we use `Re.findall_groups`, which returns the list of the strings matched by
the groups for every match.  See below for an example.

In [ ]:
let tokenize (s : string) : string list =
  (* Transform the string s into a list of tokens.  The string s
     is supposed to represent an arithmetic expression. *)
  let lexSpec = {|([ \t\n]+)      |  # blanks and tabs
                  ([1-9][0-9]*|0) |  # number
                  ([-+*/()])      |  # arithmetical operators
                  (.)                # unrecognized character
               |}
  in
  List.filter_map
    (function
      | [ ws; number; operator; error ] ->
          if ws <> "" then None                      (* skip blanks and tabs *)
          else if number <> "" then Some "NUMBER"
          else if operator <> "" then Some operator
          else Some (Printf.sprintf "ERROR(%s)" error)
      | _ -> assert false)
    (Re.findall_groups ~flags:Re.[VERBOSE] lexSpec s)

In [ ]:
tokenize "11 + 22 * (33 - 45) / 007"

The types of rules, actions, and parse tables are defined in the notebook `Parse-Table.ipynb`, which also contains the
parse table that we use below.  As OCaml must know these types before they are used, we load this notebook here.

In [ ]:
run_notebook "Parse-Table.ipynb"
;;
type action_table = (state * token, action) Hashtbl.t
type goto_table   = (state * variable, state) Hashtbl.t

Assume a grammar $G = \langle V, T, R, S \rangle$ is given.  A  *shift-reduce parser*
is defined as a 4-Tuple
$$P = \langle Q, q_0, \texttt{action}, \texttt{goto} \rangle$$
where
- $Q$ is the set of *states* of the shift-reduce parser.  

  For the purpose of the shift-reduce-parser, states are purely abstract. 
- $q_0 \in Q$ is the *start state*.
- $\texttt{action}$ is a function taking two arguments. The first argument is a state $q \in Q$
  and the second argument is a token $t \in T$.  The result of this function is an element from the set
  $$\texttt{Action} :=
       \bigl\{ \langle\texttt{shift}, q\rangle  \mid q \in Q \bigr\}               \cup 
       \bigl\{ \langle\texttt{reduce}, r\rangle \mid r \in R \bigr\} \cup 
       \bigl\{ \texttt{accept} \bigr\}                        \cup
       \bigl\{ \texttt{error}  \bigr\}.                         
  $$
  In OCaml, `Shift q`, `Reduce r`, and `Accept` are the constructors of the type `action`, while `error` is
  represented by a missing entry in the table:
  $$\texttt{action}: Q \times T \rightarrow \texttt{Action}.$$
- `goto` is a function that takes a state $q \in Q$ and a syntactical variable
  $v \in V$ and computes a new state:
  $$\texttt{goto}: Q \times V \rightarrow Q.$$

The module `ShiftReduceParser` defines the record type `t`, which contains two tables:
- `mActionTable` encodes the function $\texttt{action}: Q \times T \rightarrow \texttt{Action}$.
- `mGotoTable` encodes the function $\texttt{goto}: Q \times V \rightarrow Q$.

The function `make` takes these tables as arguments and stores them in a record.  It corresponds to the constructor of
the Python class `ShiftReduceParser`.

In [ ]:
module ShiftReduceParser = struct
  type t = { mActionTable : action_table; mGotoTable : goto_table }

  let make (actionTable : action_table) (gotoTable : goto_table) : t =
    { mActionTable = actionTable; mGotoTable = gotoTable }
end

The function `parse` takes a list of tokens `tl` as its argument.  It returns `true` if the token list can be parsed successfully or `false` otherwise.  
The algorithm that is applied is known as *shift/reduce parsing*.

The Python version adds the method `parse` to the class `ShiftReduceParser`.  In OCaml, we *extend* the module
`ShiftReduceParser`: the new module *includes* everything from the old module and adds the function `parse`.

The stacks of symbols and states are lists whose *first* element is the top of the stack, and the `while` loop of the
Python version is replaced by the recursive function `loop`.  `drop n l` removes the first `n` elements of the list `l`.

In [ ]:
module ShiftReduceParser = struct
  include ShiftReduceParser

  let rec drop (n : int) (l : 'a list) : 'a list = if n = 0 then l else drop (n - 1) (List.tl l)

  let parse (self : t) (tl : token list) : bool =
    let tl = Array.of_list (tl @ [ "EOF" ]) in
    let rec loop (index : int) (symbols : symbol list) (states : state list) : bool =
      let q = List.hd states in
      let t = tl.(index) in
      (* Any undefined table entries are interpreted as error entries. *)
      match Hashtbl.find_opt self.mActionTable (q, t) with
      | None -> false
      | Some Accept -> true
      | Some (Shift s) -> loop (index + 1) (t :: symbols) (s :: states)
      | Some (Reduce (head, body)) ->
          let n       = List.length body in
          let symbols = head :: drop n symbols in
          let states  = drop n states in
          let state   = List.hd states in
          loop index symbols (Hashtbl.find self.mGotoTable (state, head) :: states)
    in
    loop 0 [] [ "s0" ]
end

## Testing

In [ ]:
run_notebook "Parse-Table.ipynb"

**Note:** The function `run_notebook` executes all cells of a notebook, just like the magic command `%run`.  Since
the notebook `Parse-Table.ipynb` has already been loaded above, it is not loaded a second time.

In [ ]:
let test (s : string) : unit =
  let parser = ShiftReduceParser.make actionTable gotoTable in
  let tl     = tokenize s in
  if ShiftReduceParser.parse parser tl then print_endline "Parse successful!"
  else print_endline "Parse failed!"

In [ ]:
test "(1 + 2) * 3"

In [ ]:
test "1 * 2 + 3 * (4 - 5) / 2"

In [ ]:
test "11+22*(33-44)/(5-10*5/(4-3))"

In [ ]:
test "1+2*3-"

In [ ]:
test "1+2*3-007"